# Transformer kernel — pass 2 verification (Tesla T4)

Validates the `claude/integration-pass2` branch ([PR #10](https://github.com/danielfodgaard/transformer-kernel/pull/10)):
fused Triton residual+LayerNorm kernels, CUDA-graph capture, `--fp32-reductions`,
data-driven dispatch, and the case-14 out-of-core runner — plus verification of the
d32→fp32 dispatch and case-6 follow-ups from the merged pass-2 results.

**Run with Save & Run All (batch); total ≈ 1.5–2 h.** Gates run first: if cell 3
(Triton numerics) fails, every later sweep would need `--no-fused-norm` — stop and
report instead. Every measurement writes `results/*.json`; the last cell bundles them.


In [ ]:
!nvidia-smi --query-gpu=name,temperature.gpu,clocks.sm --format=csv
!cd /kaggle/working && rm -rf transformer-kernel && \
  git clone -q -b claude/integration-pass2 https://github.com/danielfodgaard/transformer-kernel.git
import torch
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


## Gates


In [ ]:
# GATE 1: Triton kernel numerics (unit level + fused-vs-eager end to end)
!cd /kaggle/working/transformer-kernel && python src/test_kernels.py


In [ ]:
# GATE 2: padded input must fall back to the eager masked path and pass accuracy
!cd /kaggle/working/transformer-kernel && python src/run_case.py --causal --batch-size 16 --d-model 128 --heads 4 \
  --seq-len 128 --layers 4 --ffn-dim 128 --padding-ratio 0.3 --warmup 0 --repeats 1 --benchmark-rounds 1


## Headline: pass-2 defaults vs the committed pass-1 numbers

Same 13 shapes as `results/pass1-fp16.json` (in the repo). New defaults in play:
fused Triton norms, memoized mask check, d32→fp32 dispatch (case 7 expects ~1e-6 error now).


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --skip 14 --out results/pass2-default.json


## Ablations and alternatives


In [ ]:
# What the fused Triton kernels contribute (cases where elementwise traffic matters)
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,5,13 --out results/pass2-nofused.json -- --no-fused-norm


In [ ]:
# Manual CUDA graphs vs torch.compile on the launch-bound shapes — never combined
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,2,3,4,12 --out results/pass2-cg.json -- --cuda-graphs
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,2,3,4,12 --out results/pass2-compiled-small.json -- --compile-user --compile-mode reduce-overhead


## Accuracy verification

Benchmark minimized (`--warmup 0 --repeats 1 --benchmark-rounds 1`) — these cells only test accuracy.
Case 7 should now sit at ~2e-6 (fp32 dispatch). Case 6 is the next-thinnest margin (0.00187 at 5 trials);
`--fp32-reductions` is the candidate knob if its 25-trial stress fails.


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 --out results/case7-stress-fp32-a.json -- --accuracy-trials 25 --warmup 0 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 --out results/case7-stress-fp32-b.json -- --accuracy-trials 25 --seed 9999 --warmup 0 --repeats 1 --benchmark-rounds 1


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/case6-stress.json -- --accuracy-trials 25 --warmup 0 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/case6-stress-fp32red.json -- --accuracy-trials 25 --fp32-reductions --warmup 0 --repeats 1 --benchmark-rounds 1


In [ ]:
# Perf cost of --fp32-reductions where GEMMs dominate
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 8 --out results/pass2-fp32red.json -- --fp32-reductions


## Case 6 + compile, without CUDA-graph memory pools

`reduce-overhead` OOMs beside the baseline at batch 10000; `default` mode fuses without graph pools.


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/case6-compile-default.json -- --compile-user --compile-mode default


## Data-driven dispatch

Generate the table from everything measured so far (committed results + this session's), then spot-check.


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/dispatch.py
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,2,12 --out results/pass2-dispatch.json -- --dispatch


## Case 14 (out of core) — heavy cells last

Dry run first at a feasible length; the full run compares chunked fp16 against the fp32 proxy
reference (itself validated against the true baseline at seq 1024).


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/run_case14.py --seq-len 4096 --max-samples 4 --out results/case14-dry.json


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/run_case14.py --out results/case14-full.json


## Summary and bundle


In [ ]:
import json, pathlib
for path in sorted(pathlib.Path('/kaggle/working/transformer-kernel/results').glob('*.json')):
    data = json.loads(path.read_text())
    if 'cases' in data:
        print(f"\n=== {path.name} | {data.get('passthrough_args')}")
        for case in data['cases']:
            acc = case.get('accuracy') or {}
            print(f"  case {case['case']['id']:>2} {case['status']:<16} "
                  f"speedup={case.get('speedup')} max_abs={acc.get('max_abs_error')}")
    else:
        print(f"\n=== {path.name}\n{json.dumps(data, indent=2)[:1500]}")


In [ ]:
!cd /kaggle/working/transformer-kernel && tar czf /kaggle/working/results.tar.gz results/
print('Download results.tar.gz from the notebook Output tab')
